# Capstone 9.2 — An Async API Aggregator

### Python Mastery · Track 9: Capstone Projects

This capstone integrates asynchronous programming (Track 5) with web skills (Module 8.2) into a realistic build: an **aggregator** that fetches data from several services at once, merges the results, and behaves well under real-world conditions with caching, retries, and timeouts. It is the pattern behind dashboards, price comparison tools, and any service that combines multiple upstream APIs.

**How to use this notebook**

- Read each section, then run the code cell beneath it. To stay fully reproducible with no external network, the notebook starts **several small mock API services on your own machine**; the aggregator then makes genuine concurrent HTTP calls to them. Run the setup cell in Stage 0 first.
- The async cells use top-level `await`, as established in Track 5 (`asyncio.run` does not work inside a notebook).
- Extension challenges with solutions appear near the end.

### What you will build

An aggregator that, given a list of upstream services, fetches them **concurrently**, merges their responses into one combined result, and adds:

- **Timeouts**, so one slow service cannot stall everything.
- **Retries with backoff**, to ride out transient failures.
- **A cache**, so repeated requests for the same resource avoid redundant calls.
- **Graceful degradation**, returning partial results when a service fails.

### Learning objectives

After completing this capstone you will be able to:

1. Fetch multiple resources concurrently with `httpx` and `asyncio.gather`.
2. Apply per-request timeouts and bounded concurrency.
3. Implement retries with exponential backoff for transient errors.
4. Add a simple cache to avoid redundant requests.
5. Aggregate partial results, degrading gracefully when a service fails.

**Prerequisites:** Track 5, and Module 8.2.

---

## Stage 0 · Start Several Mock Services

A real aggregator talks to several independent APIs. We simulate that with a few small services on localhost, each returning different data after a short, deliberate delay (to make concurrency observable). One service is configured to be flaky, failing intermittently, so we can exercise retries. Run this cell once; the services run in the background.

In [ ]:
import threading, json, http.server, socketserver, time, random

random.seed(0)   # deterministic flakiness for reproducibility

def start_service(name, payload, delay=0.1, fail_rate=0.0):
    """Start a mock API on a free port; returns the base URL."""
    state = {"calls": 0}

    class Handler(http.server.BaseHTTPRequestHandler):
        def do_GET(self):
            state["calls"] += 1
            time.sleep(delay)                         # simulate work/latency
            # Optionally fail a fraction of the time to exercise retries.
            if fail_rate and random.random() < fail_rate:
                self.send_response(503)               # Service Unavailable
                self.end_headers()
                self.wfile.write(b'{"error": "temporarily unavailable"}')
                return
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            body = dict(payload)
            body["service"] = name
            self.wfile.write(json.dumps(body).encode())
        def log_message(self, *a):
            pass

    httpd = socketserver.TCPServer(("127.0.0.1", 0), Handler)
    port = httpd.server_address[1]
    threading.Thread(target=httpd.serve_forever, daemon=True).start()
    return f"http://127.0.0.1:{port}"

# Three services: a fast one, a slow one, and a flaky one.
SERVICES = {
    "weather": start_service("weather", {"temp_c": 22, "conditions": "clear"}, delay=0.10),
    "news":    start_service("news", {"headlines": ["Market up", "Rain expected"]}, delay=0.20),
    "stocks":  start_service("stocks", {"index": "NIFTY", "change": "+0.8%"}, delay=0.15, fail_rate=0.5),
}
print("mock services running:")
for name, url in SERVICES.items():
    print(f"  {name}: {url}")

## Stage 1 · A Single Async Fetch

The building block is one asynchronous function that fetches a single service and returns its parsed JSON, with a timeout so it cannot hang. Because it is a coroutine, many of these can run concurrently later. We test it against one service first.

In [ ]:
import httpx

async def fetch_one(client, name, url):
    """Fetch one service's data, returning (name, data) or raising on failure."""
    response = await client.get(url, timeout=2.0)     # per-request timeout
    response.raise_for_status()                       # turn 4xx/5xx into an error
    return name, response.json()

# Test against a single reliable service.
async with httpx.AsyncClient() as client:
    name, data = await fetch_one(client, "weather", SERVICES["weather"])
print(f"{name} returned:", data)

## Stage 2 · Concurrent Aggregation

Now the core value: fetch all services **at the same time** rather than one after another, using `asyncio.gather` (Track 5). The total time becomes roughly that of the slowest service, not the sum of all of them. We aggregate the reliable services here and time it to prove the overlap.

In [ ]:
import httpx, asyncio, time

async def aggregate(services):
    """Fetch all services concurrently and merge into one dict keyed by service name."""
    async with httpx.AsyncClient() as client:
        tasks = [fetch_one(client, name, url) for name, url in services.items()]
        results = await asyncio.gather(*tasks)
    return {name: data for name, data in results}

# Use only the reliable services for this timing demonstration.
reliable = {k: SERVICES[k] for k in ["weather", "news"]}

start = time.perf_counter()
combined = await aggregate(reliable)
elapsed = time.perf_counter() - start

print("combined result:")
for name, data in combined.items():
    print(f"  {name}: {data}")
print(f"\nfetched concurrently in about {elapsed:.2f}s")
print("(sequential would have taken about 0.10 + 0.20 = 0.30s; concurrent is ~0.20s)")

## Stage 3 · Retries with Backoff

The `stocks` service fails about half the time, like a real flaky upstream. A robust aggregator retries transient failures with **exponential backoff** (Module 5.6): wait a little, then a little more, before giving up. We wrap the single fetch with retry logic. Transient errors (a 503, a timeout) are retried; a definitive error would not be.

In [ ]:
import httpx, asyncio

async def fetch_with_retries(client, name, url, max_attempts=5, base_delay=0.05):
    """Fetch one service, retrying transient failures with exponential backoff."""
    for attempt in range(max_attempts):
        try:
            response = await client.get(url, timeout=2.0)
            response.raise_for_status()
            return name, response.json()
        except (httpx.HTTPStatusError, httpx.TimeoutException) as e:
            if attempt == max_attempts - 1:
                raise                                    # out of attempts
            delay = base_delay * (2 ** attempt)          # 0.05, 0.10, 0.20, ...
            await asyncio.sleep(delay)                    # async wait, loop stays free

# The flaky stocks service now succeeds thanks to retries.
async with httpx.AsyncClient() as client:
    name, data = await fetch_with_retries(client, "stocks", SERVICES["stocks"])
print(f"{name} eventually returned:", data)
print("retries rode out the intermittent 503 failures")

## Stage 4 · Graceful Degradation

Sometimes a service is genuinely down and retries are exhausted. A good aggregator should still return what it *can*, marking the failed service rather than failing the whole request. We use `asyncio.gather(..., return_exceptions=True)` so one failure does not cancel the others, then separate successes from failures.

In [ ]:
import httpx, asyncio

async def aggregate_resilient(services):
    """Fetch all services concurrently with retries; return partial results on failure."""
    async with httpx.AsyncClient() as client:
        tasks = [fetch_with_retries(client, name, url, max_attempts=2)
                 for name, url in services.items()]
        # return_exceptions=True: a failure becomes a result, not a cancellation.
        outcomes = await asyncio.gather(*tasks, return_exceptions=True)

    combined, failed = {}, []
    for outcome in outcomes:
        if isinstance(outcome, Exception):
            failed.append(type(outcome).__name__)
        else:
            name, data = outcome
            combined[name] = data
    return combined, failed

# Run against all services, including the flaky one with only 2 attempts allowed,
# so it may sometimes still fail and demonstrate graceful degradation.
combined, failed = await aggregate_resilient(SERVICES)
print("services that returned data:", list(combined))
print("services that failed (if any):", failed)
print("the aggregator returns whatever succeeded rather than failing entirely")

## Stage 5 · Adding a Cache

Repeatedly fetching the same resource wastes time and load. A **cache** stores a result so the next request for it returns immediately. We add a simple in-memory cache keyed by service name, with the lesson from earlier tracks that a real cache would also track freshness (a time-to-live). The second aggregation hits the cache and is essentially instant.

In [ ]:
import httpx, asyncio, time

class AggregatorWithCache:
    """An aggregator that caches each service's last successful result."""
    def __init__(self, services):
        self.services = services
        self.cache = {}

    async def get(self, force_refresh=False):
        async with httpx.AsyncClient() as client:
            tasks = []
            cached_names = []
            for name, url in self.services.items():
                if not force_refresh and name in self.cache:
                    cached_names.append(name)           # already have it
                else:
                    tasks.append(fetch_with_retries(client, name, url, max_attempts=3))
            fetched = await asyncio.gather(*tasks, return_exceptions=True)

        for outcome in fetched:
            if not isinstance(outcome, Exception):
                name, data = outcome
                self.cache[name] = data                 # store for next time
        return {name: self.cache[name] for name in self.services if name in self.cache}, cached_names

agg = AggregatorWithCache({k: SERVICES[k] for k in ["weather", "news"]})

start = time.perf_counter()
first, _ = await agg.get()
first_time = time.perf_counter() - start

start = time.perf_counter()
second, from_cache = await agg.get()           # all served from cache
second_time = time.perf_counter() - start

print(f"first call (network):  {first_time:.3f}s, services: {list(first)}")
print(f"second call (cached):  {second_time:.4f}s, served from cache: {from_cache}")
print("the cached call avoids the network entirely and is far faster")

## Assembling the Project

The cell below assembles the aggregator into a reusable module with a clean public class. It contains no notebook-specific code; you provide a mapping of service names to URLs and call `aggregate`. This is the deliverable you can drop into a real project.

In [ ]:
import os, textwrap

os.makedirs("aggregator_project", exist_ok=True)

module = textwrap.dedent("""
    # An async API aggregator with timeouts, retries, caching, and graceful degradation.
    import asyncio
    import httpx

    class Aggregator:
        def __init__(self, services, timeout=2.0, max_attempts=3, base_delay=0.05):
            self.services = services           # {name: url}
            self.timeout = timeout
            self.max_attempts = max_attempts
            self.base_delay = base_delay
            self.cache = {}

        async def _fetch(self, client, name, url):
            for attempt in range(self.max_attempts):
                try:
                    resp = await client.get(url, timeout=self.timeout)
                    resp.raise_for_status()
                    return name, resp.json()
                except (httpx.HTTPStatusError, httpx.TimeoutException):
                    if attempt == self.max_attempts - 1:
                        raise
                    await asyncio.sleep(self.base_delay * (2 ** attempt))

        async def aggregate(self, use_cache=True):
            # Fetch all services concurrently; return (combined, failed).
            async with httpx.AsyncClient() as client:
                tasks, cached = [], {}
                for name, url in self.services.items():
                    if use_cache and name in self.cache:
                        cached[name] = self.cache[name]
                    else:
                        tasks.append(self._fetch(client, name, url))
                outcomes = await asyncio.gather(*tasks, return_exceptions=True)

            combined, failed = dict(cached), []
            for outcome in outcomes:
                if isinstance(outcome, Exception):
                    failed.append(type(outcome).__name__)
                else:
                    name, data = outcome
                    combined[name] = data
                    self.cache[name] = data
            return combined, failed
""").lstrip()

with open("aggregator_project/aggregator.py", "w") as f:
    f.write(module)

readme = textwrap.dedent("""
    # Async API Aggregator

    Fetches several HTTP services concurrently and merges their JSON responses,
    with per-request timeouts, retries with exponential backoff, an in-memory
    cache, and graceful degradation (partial results when a service fails).

    ## Usage
    ```python
    import asyncio
    from aggregator import Aggregator

    services = {"weather": "http://...", "news": "http://..."}
    agg = Aggregator(services)
    combined, failed = asyncio.run(agg.aggregate())
    ```
""").lstrip()
with open("aggregator_project/README.md", "w") as f:
    f.write(readme)

print("project assembled:")
for name in sorted(os.listdir("aggregator_project")):
    print("  aggregator_project/" + name)

In [ ]:
# Verify the assembled module works by importing and running it against our live mock services.
import sys, importlib
sys.path.insert(0, "aggregator_project")
import aggregator as agg_module
importlib.reload(agg_module)

agg = agg_module.Aggregator({k: SERVICES[k] for k in ["weather", "news"]})
combined, failed = await agg.aggregate()
print("assembled Aggregator returned:", list(combined))
print("failed:", failed)
print("the packaged class works against real concurrent HTTP calls")

---

## Demonstrated Variations

### Variation 1 — Bounding concurrency with a semaphore

In [ ]:
import httpx, asyncio
# When there are many services, cap how many run at once (Module 5.5).
async def aggregate_bounded(services, limit=2):
    sem = asyncio.Semaphore(limit)
    async def guarded(client, name, url):
        async with sem:
            return await fetch_one(client, name, url)
    async with httpx.AsyncClient() as client:
        tasks = [guarded(client, n, u) for n, u in services.items()]
        results = await asyncio.gather(*tasks)
    return {n: d for n, d in results}

out = await aggregate_bounded({k: SERVICES[k] for k in ["weather", "news"]}, limit=1)
print("bounded aggregation:", list(out))

### Variation 2 — Forcing a cache refresh

In [ ]:
agg2 = AggregatorWithCache({k: SERVICES[k] for k in ["weather"]})
await agg2.get()                                  # populate cache
fresh, from_cache = await agg2.get(force_refresh=True)   # bypass cache
print("forced refresh re-fetched:", list(fresh), "| from cache:", from_cache)

### Variation 3 — A combined summary view

In [ ]:
combined, _ = await aggregate_resilient({k: SERVICES[k] for k in ["weather", "news"]})
# Build a single human-readable summary from the merged data.
summary = []
if "weather" in combined:
    summary.append(f"Weather: {combined['weather']['temp_c']}C, {combined['weather']['conditions']}")
if "news" in combined:
    summary.append(f"Top headline: {combined['news']['headlines'][0]}")
print("\n".join(summary))

---

## Extension Challenges

**Challenge 1.** Add a per-service timing measurement to the aggregator so it reports how long each service took. Demonstrate it on two services.

In [ ]:
# Your solution here


**Challenge 2.** Add a time-to-live (TTL) to the cache: a cached entry older than `ttl` seconds should be refetched. Demonstrate that a fresh entry is served from cache but an expired one is refetched.

In [ ]:
# Your solution here


**Challenge 3.** Modify `aggregate_resilient` to also return, for each failed service, the name of the service (not just the exception type), so callers know exactly what is missing.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
import httpx, asyncio, time

async def fetch_timed(client, name, url):
    start = time.perf_counter()
    resp = await client.get(url, timeout=2.0)
    resp.raise_for_status()
    return name, resp.json(), time.perf_counter() - start

async with httpx.AsyncClient() as client:
    tasks = [fetch_timed(client, n, SERVICES[n]) for n in ["weather", "news"]]
    for name, data, took in await asyncio.gather(*tasks):
        print(f"{name}: {took:.2f}s")

**Solution 2**

In [ ]:
import time, asyncio, httpx

class TTLCache:
    def __init__(self, services, ttl=0.5):
        self.services, self.ttl, self.cache = services, ttl, {}
    async def get(self, name):
        now = time.time()
        if name in self.cache and now - self.cache[name][1] < self.ttl:
            return self.cache[name][0], "cache"
        async with httpx.AsyncClient() as client:
            resp = await client.get(self.services[name], timeout=2.0)
        self.cache[name] = (resp.json(), now)
        return resp.json(), "network"

c = TTLCache({"weather": SERVICES["weather"]}, ttl=0.5)
_, src1 = await c.get("weather")
_, src2 = await c.get("weather")           # fresh -> cache
await asyncio.sleep(0.6)
_, src3 = await c.get("weather")           # expired -> network
print("first:", src1, "| immediate repeat:", src2, "| after TTL:", src3)

**Solution 3**

In [ ]:
import httpx, asyncio

async def aggregate_named_failures(services):
    async with httpx.AsyncClient() as client:
        names = list(services)
        tasks = [fetch_with_retries(client, n, services[n], max_attempts=2) for n in names]
        outcomes = await asyncio.gather(*tasks, return_exceptions=True)
    combined, failed = {}, []
    for name, outcome in zip(names, outcomes):
        if isinstance(outcome, Exception):
            failed.append(name)                  # the service name, not just the error type
        else:
            combined[outcome[0]] = outcome[1]
    return combined, failed

combined, failed = await aggregate_named_failures(SERVICES)
print("succeeded:", list(combined), "| failed services:", failed)

---

## Summary and Key Points

- An aggregator fetches several services concurrently with `asyncio.gather`, so total time is about that of the slowest service rather than the sum, the core efficiency win from Track 5.
- Each fetch sets a per-request timeout so one slow upstream cannot stall the whole aggregation, and concurrency can be bounded with a semaphore when there are many services.
- Transient failures (503s, timeouts) are retried with exponential backoff using `await asyncio.sleep`, which waits without blocking the event loop; definitive errors are not retried.
- `asyncio.gather(..., return_exceptions=True)` enables graceful degradation: one service failing becomes a recorded result rather than cancelling the others, so callers get partial data.
- A cache avoids redundant calls (a real one adds a time-to-live for freshness); the finished aggregator is packaged as a reusable class with timeouts, retries, caching, and degradation built in.

### Next capstone: 9.3 — A CLI Productivity Tool

The final capstone builds a command-line task manager with JSON persistence, a test suite, and packaging, integrating core Python, file handling, testing, and CLI skills from across the course.